# 02 Keyword Search

Import Libraries

In [ ]:
import pandas as pd
import re
from pathlib import Path

Load cleaned data

In [ ]:
cleaned_data_path = Path(
    "../../data/processed/keyword_ready_messages.csv"
)


In [ ]:
keyword_df = pd.read_csv(cleaned_data_path)

Prevent missing text from causing matching errors

In [ ]:
keyword_df["clean_text"] = (
    keyword_df["clean_text"]
    .fillna("")
    .astype(str)
)

In [ ]:
print("Cleaned data load successed")
print('Available columns:', keyword_df.columns.tolist())

Keyword Categories

Keywords are grouped according to products, customer concerns, reasons for
contacting the chatbot, and campaign-related interest.

The initial dictionary is based on the project objectives and common terms in
loan and insurance conversations. It will be improved after reviewing matched
and unmatched messages.

In [ ]:
# Existing broad categories are preserved; detailed subcategories support drill-down analysis.
keyword_categories = {'loan_product': ['loan',
                  'car loan',
                  'motorcycle loan',
                  'land loan',
                  'refinance',
                  'refinancing',
                  'financing',
                  'credit'],
 'insurance_product': ['insurance',
                       'car insurance',
                       'motorcycle insurance',
                       'policy',
                       'coverage'],
 'car_loan': ['car loan',
              'auto loan',
              'vehicle loan',
              'car financing',
              'car finance',
              'loan for car',
              'finance a car',
              'used car loan',
              'new car loan'],
 'motorcycle_loan': ['motorcycle loan',
                     'motorbike loan',
                     'bike loan',
                     'big bike loan',
                     'motorcycle financing',
                     'loan for motorcycle',
                     'loan for motorbike'],
 'land_loan': ['land loan',
               'loan for land',
               'land financing',
               'land collateral loan',
               'loan using land',
               'land title deed',
               'title deed loan',
               'mortgage land'],
 'refinancing': ['refinance',
                 'refinancing',
                 'refinance car',
                 'car refinance',
                 'motorcycle refinance',
                 'debt refinancing',
                 'transfer debt',
                 'close old debt'],
 'car_insurance': ['car insurance',
                   'auto insurance',
                   'vehicle insurance',
                   'motor insurance for car',
                   'compulsory car insurance',
                   'voluntary car insurance',
                   'class 1 car insurance'],
 'motorcycle_insurance': ['motorcycle insurance',
                          'motorbike insurance',
                          'bike insurance',
                          'motor insurance for motorcycle',
                          'compulsory motorcycle insurance'],
 'other_insurance_product': ['personal accident insurance',
                             'accident insurance',
                             'health insurance',
                             'life insurance',
                             'travel insurance',
                             'home insurance',
                             'fire insurance',
                             'property insurance',
                             'pa insurance'],
 'eligibility': ['eligible',
                 'eligibility',
                 'qualify',
                 'qualification',
                 'salary',
                 'income',
                 'age requirement'],
 'application_process': ['apply',
                         'apply for',
                         'application',
                         'register',
                         'registration',
                         'submit',
                         'submit application'],
 'required_documents': ['document',
                        'documents',
                        'paperwork',
                        'required document',
                        'required documents',
                        'documents required',
                        'what documents',
                        'prepare documents',
                        'supporting documents',
                        'copy of id card',
                        'bank statement',
                        'salary slip',
                        'registration book'],
 'approval_and_status': ['approve',
                         'approved',
                         'approval',
                         'application status',
                         'check status',
                         'pending',
                         'rejected'],
 'interest_and_fees': ['interest', 'interest rate', 'fee', 'fees', 'charge', 'charges'],
 'credit_limit': ['credit limit',
                  'loan limit',
                  'increase the credit limit',
                  'loan amount',
                  'credit amount',
                  'approved amount',
                  'maximum loan',
                  'maximum amount',
                  'how much can i borrow'],
 'payment_and_installment': ['payment',
                             'pay',
                             'installment',
                             'monthly payment',
                             'repayment',
                             'payment channel',
                             'overdue'],
 'insurance_premium': ['premium',
                       'insurance premium',
                       'premium price',
                       'insurance price'],
 'insurance_claim': ['claim', 'make a claim', 'claim status', 'accident', 'damage'],
 'coverage_and_conditions': ['coverage',
                             'condition',
                             'conditions',
                             'requirement',
                             'requirements',
                             'benefit',
                             'benefits',
                             'protection'],
 'renewal_and_cancellation': ['renew',
                              'renewal',
                              'expired',
                              'expiration',
                              'cancel',
                              'cancellation'],
 'interest_rate': ['interest rate',
                   'interest per month',
                   'interest per year',
                   'monthly interest',
                   'annual interest',
                   'rate per month',
                   'rate per year'],
 'fees': ['service fee',
          'processing fee',
          'application fee',
          'contract fee',
          'transfer fee',
          'late fee',
          'penalty fee',
          'additional fee',
          'extra charge'],
 'loan_installment_repayment': ['loan installment',
                                'monthly installment',
                                'installment amount',
                                'installment payment',
                                'repayment',
                                'repayment schedule',
                                'pay installment',
                                'loan payment'],
 'insurance_premium_payment': ['insurance premium',
                               'premium payment',
                               'pay premium',
                               'insurance payment',
                               'premium installment',
                               'premium amount'],
 'payment_channel': ['payment channel',
                     'payment method',
                     'where to pay',
                     'how to pay',
                     'pay through',
                     'pay via',
                     'bank transfer',
                     'counter service',
                     'qr payment',
                     'mobile banking',
                     'automatic deduction'],
 'payment_status': ['payment status',
                    'payment successful',
                    'payment failed',
                    'payment not updated',
                    'paid or not',
                    'already paid',
                    'not paid yet',
                    'payment confirmation',
                    'proof of payment',
                    'receipt'],
 'overdue_payment': ['overdue',
                     'late payment',
                     'past due',
                     'missed payment',
                     'outstanding balance',
                     'arrears',
                     'debt collection',
                     'payment overdue'],
 'insurance_coverage': ['insurance coverage',
                        'coverage',
                        'cover damage',
                        'covered damage',
                        'what is covered',
                        'sum insured',
                        'deductible',
                        'third party',
                        'repair coverage'],
 'renewal': ['renew',
             'renewal',
             'renew policy',
             'renew insurance',
             'policy renewal',
             'extend policy',
             'expired policy'],
 'cancellation': ['cancel',
                  'cancellation',
                  'cancel policy',
                  'cancel insurance',
                  'cancel application',
                  'terminate policy',
                  'close account'],
 'approval_application_status': ['application status',
                                 'approval status',
                                 'check status',
                                 'check application',
                                 'application result',
                                 'approved',
                                 'not approved',
                                 'pending approval',
                                 'rejected'],
 'requesting_information': ['want to know',
                            'would like to know',
                            'ask about',
                            'inquire about',
                            'information about',
                            'details about',
                            'please tell me',
                            'please advise',
                            'more details'],
 'applying': ['apply for',
              'want to apply',
              'would like to apply',
              'submit application',
              'start application',
              'register for',
              'sign up for'],
 'checking_eligibility': ['am i eligible',
                          'can i apply',
                          'can i borrow',
                          'do i qualify',
                          'check eligibility',
                          'eligible for',
                          'qualify for',
                          'income requirement',
                          'salary requirement'],
 'checking_status': ['check status',
                     'application status',
                     'approval status',
                     'payment status',
                     'claim status',
                     'follow up application',
                     'follow up status'],
 'making_payment': ['make a payment',
                    'want to pay',
                    'need to pay',
                    'pay installment',
                    'pay premium',
                    'pay the bill',
                    'pay via',
                    'where to pay',
                    'how to pay'],
 'making_claim': ['make a claim',
                  'file a claim',
                  'submit claim',
                  'claim process',
                  'claim documents',
                  'claim status',
                  'report accident',
                  'accident claim'],
 'renewing': ['renew insurance',
              'renew policy',
              'policy renewal',
              'extend policy',
              'renew coverage'],
 'cancelling': ['cancel insurance',
                'cancel policy',
                'cancel application',
                'terminate policy',
                'cancel service'],
 'contacting_staff': ['contact staff',
                      'talk to staff',
                      'speak with staff',
                      'speak with officer',
                      'talk to officer',
                      'contact officer',
                      'call center',
                      'human staff',
                      'staff contact',
                      'please call me'],
 'finding_branch': ['branch',
                    'branch location',
                    'nearest branch',
                    'where is the branch',
                    'find branch',
                    'service center',
                    'office location'],
 'campaign_or_promotion': ['campaign',
                           'promotion',
                           'promotional',
                           'offer',
                           'special offer',
                           'discount',
                           'privilege',
                           'reward'],
 'campaign_interest': ['campaign interest',
                       'interested in campaign',
                       'interested in promotion',
                       'join campaign',
                       'participate in campaign',
                       'get the offer',
                       'receive the right',
                       'receive rights'],
 'campaign_eligibility': ['campaign eligibility',
                          'eligible for campaign',
                          'eligible for promotion',
                          'promotion eligibility',
                          'qualify for campaign',
                          'qualify for promotion',
                          'campaign condition',
                          'promotion condition'],
 'campaign_registration': ['campaign registration',
                           'promotion registration',
                           'register campaign',
                           'register for campaign',
                           'register promotion',
                           'register for promotion',
                           'join campaign',
                           'participate in campaign',
                           'sign up for campaign'],
 'campaign_benefits': ['campaign benefit',
                       'campaign benefits',
                       'promotion benefit',
                       'promotion benefits',
                       'campaign privilege',
                       'promotion privilege',
                       'discount',
                       'cashback',
                       'reward',
                       'special offer'],
 'campaign_period_deadline': ['campaign period',
                              'promotion period',
                              'campaign deadline',
                              'promotion deadline',
                              'last day',
                              'end date',
                              'valid until',
                              'expire date',
                              'deadline'],
 'branch_or_contact': ['branch',
                       'location',
                       'contact',
                       'phone number',
                       'call me',
                       'contact me']}

Create matching function
-- The function uses word boundaries to reduce patial-word flase matches.

In [ ]:
def find_keyword_matches(text, keywords):
    matches = []

    for keyword in keywords:
        pattern = rf"(?<!\w){re.escape(keyword)}(?!\w)"

        if re.search(pattern, text, flags=re.IGNORECASE):
            matches.append(keyword)

    return matches

Text fun_: with artifical text

In [ ]:
test_message = "what documents do i need to apply for a car loan"

for category, keywords in keyword_categories.items():
    matches = find_keyword_matches(test_message, keywords)

    if matches:
        print(f"{category}: {matches}")

Apply keyword matching

Match list column for each category

In [ ]:
for category, keywords in keyword_categories.items():
    keyword_df[f"{category}_matches"] = keyword_df["clean_text"].apply(
        lambda text: find_keyword_matches(text, keywords)
    )

Boolean category columns

In [ ]:
for category in keyword_categories:
    keyword_df[category] = keyword_df[
        f"{category}_matches"
    ].apply(bool)

Record all matched categories and keywords

In [ ]:
def get_matched_categories(row):
    return[
        category
        for category in keyword_categories
        if row[category]
    ]

def get_all_matched_keywords(row):
    results = []

    for category in keyword_categories:
        for keyword in row[f"{category}_matches"]:
            results.append(f'{category}:{keyword}')

    return results

In [ ]:
keyword_df['matched_categories'] = keyword_df.apply(
    get_matched_categories,
    axis=1
)

keyword_df['matched_keywords'] = keyword_df.apply(
    get_all_matched_keywords,
    axis=1
)

keyword_df["has_keyword_match"] = keyword_df[
    "matched_categories"
].apply(bool)

Check the overall matching result

In [ ]:
matched_count = keyword_df["has_keyword_match"].sum()
unmatched_count = (~keyword_df["has_keyword_match"]).sum()
total_messages = len(keyword_df)

matched_percentage = matched_count / total_messages * 100
unmatched_percentage = unmatched_count / total_messages * 100

print(f"Matched messages: {matched_count:,} ({matched_percentage:.2f}%)")
print(f"Unmatched messages: {unmatched_count:,} ({unmatched_percentage:.2f}%)")

Category summary

In [ ]:
category_summary = pd.DataFrame({
    "category": list(keyword_categories.keys()),
    "message_count": [
        keyword_df[category].sum()
        for category in keyword_categories
    ]
})

category_summary["percentage"] = (
    category_summary["message_count"]
    / len(keyword_df)
    * 100
).round(2)

category_summary = category_summary.sort_values(
    "message_count",
    ascending=False
).reset_index(drop=True)

category_summary

Validate matched messages

In [ ]:
matched_sample = keyword_df.loc[
    keyword_df["has_keyword_match"],
    [
        "matched_categories",
        "matched_keywords"
    ]
].sample(
    n=min(20, matched_count),
    random_state=42
)

matched_sample

Review unmatched messages

In [ ]:
unmatched_df = keyword_df.loc[
    ~keyword_df["has_keyword_match"]
].copy()

unmatched_review_note = pd.DataFrame({
    "validation_item": ["Unmatched-message review"],
    "note": [
        "Inspect unmatched_df locally when needed; customer message text is not displayed in saved notebook outputs."
    ]
})

unmatched_review_note

Save the keyword results locally

In [ ]:
keyword_results = keyword_df.copy()

list_columns = [
    "matched_categories",
    "matched_keywords"
] + [
    f"{category}_matches"
    for category in keyword_categories
]

for column in list_columns:
    keyword_results[column] = keyword_results[column].apply(
        lambda values: " | ".join(values)
    )

In [ ]:
output_path = Path(
    "../../data/processed/keyword_search_results.csv"
)

keyword_results.to_csv(output_path, index=False)

print("Keyword results saved successfully.")